# 19 — Fannie multi-year model reliability
## Stage 1 — Input and frozen model verification
Repeat the approved preflight checks before scoring. Keep the two primary 22-feature models frozen. No FICO sensitivity or model updates.

In [ ]:
from pathlib import Path
import hashlib, json, sys, platform
import pandas as pd
import numpy as np
import sklearn, xgboost, joblib

ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
FANNIE = ROOT / 'data/processed/Fannie_Mae'
FREDDIE = ROOT / 'data/processed/Freddie_Mac'
COMPANION = FREDDIE / 'companion_models_2015_2017/20261125T142258634663Z'

def sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

matches = [p for p in (FANNIE / 'notebook_18d_consolidated_v1/final').glob('*/manifest.json')
           if sha(p) == '41125a32393ce4170dbd9ee75399da6b616975298cbb92dd145a6ebad5f24fbc']
assert matches, 'Reviewed Notebook 18D manifest not found.'
manifest_path = sorted(matches)[0]
m18 = json.loads(manifest_path.read_text())
DATA = manifest_path.parent / 'fannie_2015_2023_sampled_dataset.parquet'
assert sha(DATA) == m18['dataset_sha256'], 'Dataset checksum differs.'
assert m18['archive_count'] == 40 and m18['rows'] == 450000
assert not m18['fico_sensitivity_approved']
print('Verified dataset:', DATA)
print('SHA-256:', m18['dataset_sha256'])
print('Release:', m18['release_label'])

## Check the frozen Freddie Mac models
Use the reviewed Notebook 16 run and unchanged feature and evaluation sources.

In [ ]:
p = COMPANION / 'model_manifest.json'
assert sha(p) == '0831ffa3f152a46c0d6745cec758fd3dc98bf6b416dc57b3d9ccb8461afbfac7'
m16 = json.loads(p.read_text())
versions = dict(pandas=pd.__version__, numpy=np.__version__, scikit_learn=sklearn.__version__,
                xgboost=xgboost.__version__, joblib=joblib.__version__)
for name, value in versions.items():
    assert value == m16[name], f'Environment differs: {name}'
for name, expected in m16['source_sha256'].items():
    assert sha(ROOT / name) == expected, f'Source changed: {name}'
assert sha(ROOT / 'src/fannie_external_evaluation.py') == '0c380257957742615c7aa3d3fb97c70039cae5526151548f8b8cc809da9235c4'
from src.freddie_companion import apply_contract
from src.fannie_external_evaluation import input_coverage
keys = ['primary_22__logistic', 'primary_22__xgboost']
features = m16['specifications']['primary_22']
assert len(features) == 22 and set(features) == set(m18['primary_features'])
models, contracts = {}, {}
for key in keys:
    r = m16['models'][key]
    assert sha(COMPANION / r['artifact']) == r['sha256']
    assert sha(COMPANION / r['feature_contract']) == r['feature_contract_sha256']
    contracts[key] = json.loads((COMPANION / r['feature_contract']).read_text())
    models[key] = joblib.load(COMPANION / r['artifact'])
    assert list(models[key].feature_names_in_) == features == contracts[key]['predictors']
    assert contracts[key]['training_years'] == [2015, 2016, 2017]
    print('Verified:', key)
for name in ['frozen_calibration_bins.json']:
    assert sha(COMPANION / name) == m16['output_sha256'][name]
print('Frozen model verification passed.')

## Check cohort counts and predictor coverage
Reconcile the complete local sample with the reviewed cohort table. Strict eligibility must be nested in primary. Report mapping conflicts before any scoring.

In [ ]:
df = pd.read_parquet(DATA)
assert len(df) == 450000 and df.loan_id.notna().all() and df.loan_id.is_unique
assert df.groupby('origination_year').size().to_dict() == dict.fromkeys(range(2015, 2024), 50000)
for policy in ['primary', 'strict']:
    assert pd.api.types.is_bool_dtype(df[policy]) and df[policy].notna().all()
    y = df[f'TARGET_{policy.upper()}_COMPOSITE']
    assert y[df[policy]].isin([0, 1]).all() and y[~df[policy]].isna().all()
assert (~df.strict | df.primary).all()
assert df.loc[df.strict,'TARGET_PRIMARY_COMPOSITE'].equals(df.loc[df.strict,'TARGET_STRICT_COMPOSITE'])
assert int(df.primary.sum()) == 430569 and int(df.strict.sum()) == 430064
assert int(df.TARGET_PRIMARY_COMPOSITE.sum()) == 7223 and int(df.TARGET_STRICT_COMPOSITE.sum()) == 7178
for name in ['ORIGINAL_DTI','ORIGINAL_LTV','MI_PERCENT']:
    assert df[name + '_MISSING'].eq(df[name].isna().astype(int)).all()
rows = []
for period,g in df.groupby('half_year'):
    for policy in ['primary','strict']:
        rows.append(dict(cohort=period,policy=policy,sampled=len(g),eligible=int(g[policy].sum()),events=int(g[f'TARGET_{policy.upper()}_COMPOSITE'].sum())))
actual = pd.DataFrame(rows).sort_values(['cohort','policy']).reset_index(drop=True)
expected = pd.read_csv(manifest_path.parent / 'cohort_summary.csv')
expected = expected.loc[expected.granularity.eq('half_year'),actual.columns].sort_values(['cohort','policy']).reset_index(drop=True)
pd.testing.assert_frame_equal(actual,expected,check_dtype=False)
oot = df.loc[df.primary & df.origination_year.between(2018,2023)].copy()
assert len(oot) == 288063 and int(oot.TARGET_PRIMARY_COMPOSITE.sum()) == 6279
assert set(oot.half_year) == {f'{y}-H{h}' for y in range(2018,2024) for h in [1,2]}
print('OOT eligible:',len(oot))
print('MI mapping conflicts (all sampled):',int(df.MI_MAPPING_INCONSISTENT.sum()))
print('MI mapping conflicts (OOT eligible):',int(oot.MI_MAPPING_INCONSISTENT.sum()))
print('Origination field conflicts (OOT eligible):',int(oot.origination_field_conflict_count.gt(0).sum()))
display(actual)
coverage=[]
for period,g in oot.groupby('half_year'):
    for key in keys:
        x = apply_contract(g, contracts[key])
        assert list(x.columns) == features
        coverage.append(input_coverage(g,contracts[key],models[key]).assign(period=period,model=key))
coverage=pd.concat(coverage,ignore_index=True)
display(coverage.head())
print('Stage 1 checks completed. Review conflicts and coverage before scoring.')

## Stage 2 — Score the external population
Score the primary eligible Fannie loans once. Strict eligibility is a nested subset. Use the same frozen models and development calibration bins as Notebook 17. Keep loan-level probabilities local.

In [ ]:
from datetime import datetime, timezone
import zipfile
import matplotlib.pyplot as plt
from src.freddie_companion import calibration_table
from src.fannie_external_evaluation import METRICS, summarize, bootstrap_group, intervals, stable_seed

assert int(oot.MI_MAPPING_INCONSISTENT.sum()) == 0
assert int(oot.origination_field_conflict_count.gt(0).sum()) == 482
assert int(coverage.unseen_after_mapping.sum()) == 0
assert sha(DATA) == 'c57dbd9977b7b97e9b2fb7a50d3e6eb879a1057b256aafe1cf8ec1ecc30baf28'
edges_all = json.loads((COMPANION / 'frozen_calibration_bins.json').read_text())
edges = {k: edges_all[k] for k in keys}
RUN17_HASH = "d947cb7b7e674c8da3399fbc2236cd320a06ac49ca48d75829d88b0cf2cf785a"
matches17 = [p for p in (FANNIE / 'notebook_17_external_model_reliability').glob('*/model_evaluation_manifest.json')
             if sha(p) == RUN17_HASH]
assert matches17, 'Reviewed Notebook 17 run not found; keep its original output folder.'
p17 = sorted(matches17)[0]
m17 = json.loads(p17.read_text())
assert m17['reviewed_companion_manifest_sha256'] == sha(COMPANION / 'model_manifest.json')
reference_files = ['performance_point_estimates.csv', 'bootstrap_metric_draws.csv', 'bootstrap_metric_intervals.csv']
for name in reference_files:
    assert sha(p17.parent / name) == m17['output_sha256'][name], f'Notebook 17 output changed: {name}'
OUT = FANNIE / 'notebook_19_multi_year_model_reliability' / 'final' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True, exist_ok=False)
(OUT / 'figures').mkdir()
fannie = oot[['loan_id','origination_year','ORIGINATION_COHORT','half_year','strict','origination_field_conflict_count']].reset_index(drop=True)
fannie['y_true'] = oot.TARGET_PRIMARY_COMPOSITE.to_numpy(dtype=int)
for key in keys:
    x = apply_contract(oot, contracts[key])
    assert list(x.columns) == features
    p = models[key].predict_proba(x)[:, 1].astype(float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    fannie[key] = p
assert len(fannie) == 288063 and fannie.loan_id.is_unique
assert fannie.strict.sum() == 287643 and fannie.loc[fannie.strict,'y_true'].sum() == 6239
fannie.to_parquet(OUT / 'fannie_2018_2023_predictions.parquet', index=False)
actual.to_csv(OUT / 'cohort_reconciliation.csv', index=False)
coverage.to_csv(OUT / 'input_coverage.csv', index=False)
print('Scored:', len(fannie), 'loans. No model fitting.')

## Point estimates and calibration
Use half-year cohorts as the main analysis, with annual and quarterly summaries for context. The signed gap is predicted minus observed; negative values indicate underprediction. AP is average precision, not trapezoidal PR-AUC.

In [ ]:
periods = [f'{y}-H{h}' for y in range(2018,2024) for h in [1,2]]
parts, cal_parts, fixed_parts = [], [], []
for kind, col in [('half_year','half_year'),('annual','origination_year'),('quarter','ORIGINATION_COHORT')]:
    for period, parent in fannie.groupby(col, sort=True):
        for policy, g in [('primary',parent),('strict',parent.loc[parent.strict])]:
            y = g.y_true.to_numpy(dtype=int)
            preds = {k:g[k].to_numpy(dtype=float) for k in keys}
            meta = dict(agency='Fannie', period_kind=kind, period=str(period), eligibility=policy,
                        population=f'Fannie {period} {policy}')
            parts.append(summarize(y,preds,edges).assign(**meta))
            for k in keys:
                cal_parts.append(calibration_table(y,preds[k],edges[k]).assign(model=k,**meta))
                fixed_parts.append(calibration_table(y,preds[k]).assign(model=k,**meta))
perf = pd.concat(parts,ignore_index=True)
cal = pd.concat(cal_parts,ignore_index=True)
fixed = pd.concat(fixed_parts,ignore_index=True)
assert len(perf) == (12+6+24)*2*2
for row in perf.itertuples():
    for table in [cal,fixed]:
        b = table[(table.population==row.population)&(table.model==row.model)]
        assert b.n.sum()==row.n and b.events.sum()==row.events
perf.to_csv(OUT/'fannie_performance.csv',index=False)
cal.to_csv(OUT/'fannie_calibration_development_bins.csv',index=False)
fixed.to_csv(OUT/'fannie_calibration_fixed_bins.csv',index=False)
display(perf.loc[perf.period_kind.eq('half_year') & perf.eligibility.eq('primary')].round(6))

## Bootstrap uncertainty
Use 1,000 ordinary loan resamples per Fannie half-year, with both models sharing each draw. Derive strict results from the same parent resample. Reuse the verified Freddie companion-model draws from Notebook 17 rather than resampling them again. Intervals are marginal and conditional on fitted models; common shocks and training uncertainty are not included.

In [ ]:
N_BOOTSTRAP=1000
SEED=42
f_draws, f_ci, paired, strict_diff = [], [], [], []
for period in periods:
    g=fannie.loc[fannie.half_year.eq(period)]
    result=bootstrap_group(g.y_true.to_numpy(dtype=int),
                           {k:g[k].to_numpy(dtype=float) for k in keys},edges,
                           draws=N_BOOTSTRAP,seed=stable_seed(f'Fannie 19 {period}',SEED),
                           subset=g.strict.to_numpy(dtype=bool),
                           progress=lambda done,total,p=period:print(f'{p}: {done}/{total}',flush=True))
    assert result['models']==keys
    points={}
    for policy, draws, sizes in [('primary',result['draws'],np.full(N_BOOTSTRAP,len(g))),
                                 ('strict',result['subset_draws'],result['subset_sizes'])]:
        name=f'Fannie {period} {policy}'
        point=perf.loc[perf.population.eq(name)].set_index('model').loc[keys,METRICS].to_numpy(float)
        points[policy]=point
        for j,k in enumerate(keys):
            f_ci.append(intervals(draws[:,j],point[j],dict(population=name,model=k)))
            f_draws.append(pd.DataFrame(draws[:,j],columns=METRICS).assign(
                population=name,model=k,draw_index=np.arange(N_BOOTSTRAP),n_draw=sizes))
        paired.append(intervals(draws[:,1]-draws[:,0],point[1]-point[0],
                               dict(population=name,comparison='xgboost_minus_logistic',dependence='paired_same_loans')))
    for j,k in enumerate(keys):
        strict_diff.append(intervals(result['subset_draws'][:,j]-result['draws'][:,j],
                            points['strict'][j]-points['primary'][j],
                            dict(period=period,model=k,comparison='strict_minus_primary',dependence='nested_same_parent_resample')))
f_draws=pd.concat(f_draws,ignore_index=True)
f_ci=pd.concat(f_ci,ignore_index=True)
assert len(f_draws)==12*2*2*N_BOOTSTRAP
f_draws.to_csv(OUT/'fannie_bootstrap_metric_draws.csv',index=False)
f_ci.to_csv(OUT/'fannie_bootstrap_metric_intervals.csv',index=False)
pd.concat(paired,ignore_index=True).to_csv(OUT/'fannie_paired_model_difference_intervals.csv',index=False)
pd.concat(strict_diff,ignore_index=True).to_csv(OUT/'fannie_strict_minus_primary_intervals.csv',index=False)

## Stage 3 — Compare Freddie and Fannie reliability
Compare matched half-year cohorts using identical frozen 22-feature models. The independent-agency bootstrap contrast is descriptive, not a causal agency effect. The sampling frames, product coverage, mapped outcomes and prevalence can differ. AP and Brier are prevalence-dependent.

In [ ]:
ref=pd.read_csv(p17.parent/'performance_point_estimates.csv',float_precision='round_trip')
ref=ref.loc[ref.agency.eq('Freddie') & ref.model.isin(keys)].copy()
rd=pd.read_csv(p17.parent/'bootstrap_metric_draws.csv',float_precision='round_trip')
rd=rd.loc[rd.population.isin([f'Freddie {p}' for p in periods]) & rd.model.isin(keys)].copy()
rc=pd.read_csv(p17.parent/'bootstrap_metric_intervals.csv',float_precision='round_trip')
rc=rc.loc[rc.population.isin([f'Freddie {p}' for p in periods]) & rc.model.isin(keys)].copy()
assert len(ref)==(12+6+24)*2 and len(rd)==12*2*N_BOOTSTRAP
assert ref.loc[ref.period_kind.eq('half_year') & ref.model.eq(keys[0]),'n'].sum()==294070
combined=pd.concat([ref,perf],ignore_index=True)
combined.to_csv(OUT/'cross_agency_performance.csv',index=False)
ci=pd.concat([rc,f_ci],ignore_index=True)
ci.to_csv(OUT/'cross_agency_metric_intervals.csv',index=False)
contrast=[]
for period in periods:
    right=f'Freddie {period}'
    for policy in ['primary','strict']:
        left=f'Fannie {period} {policy}'
        for k in keys:
            a=f_draws.loc[f_draws.population.eq(left)&f_draws.model.eq(k)].sort_values('draw_index')
            b=rd.loc[rd.population.eq(right)&rd.model.eq(k)].sort_values('draw_index')
            assert np.array_equal(a.draw_index.to_numpy(),np.arange(N_BOOTSTRAP))
            assert np.array_equal(b.draw_index.to_numpy(),np.arange(N_BOOTSTRAP))
            pa=perf.loc[perf.population.eq(left)&perf.model.eq(k),METRICS].to_numpy(float)[0]
            pb=ref.loc[ref.population.eq(right)&ref.model.eq(k),METRICS].to_numpy(float)[0]
            contrast.append(intervals(a[METRICS].to_numpy(float)-b[METRICS].to_numpy(float),pa-pb,
                dict(period=period,model=k,eligibility=policy,comparison='Fannie_minus_Freddie',dependence='independent_agency_populations')))
contrasts=pd.concat(contrast,ignore_index=True)
contrasts.to_csv(OUT/'cross_agency_difference_intervals.csv',index=False)
display(contrasts.loc[contrasts.eligibility.eq('primary') & contrasts.metric.isin(['roc_auc','brier','risk_gap_pred_minus_observed'])].round(6))

## Plot temporal reliability
Show primary external results against Freddie companion-model results. The strict sensitivity remains in the tables. All plotted values and intervals come from the evaluation outputs.

In [ ]:
plot_metrics=[('roc_auc','ROC-AUC'),('average_precision','Average precision'),('brier','Brier score'),
              ('risk_gap_pred_minus_observed','Risk gap (predicted − observed)'),
              ('absolute_risk_gap','Absolute risk gap'),('ece_development_bins','Development-bin ECE')]
for k in keys:
    fig,axes=plt.subplots(3,2,figsize=(13,11))
    for ax,(metric,label) in zip(axes.flat,plot_metrics):
        for agency,policy,color in [('Freddie','original','#1f77b4'),('Fannie','primary','#d95f02')]:
            data=combined.loc[combined.agency.eq(agency)&combined.eligibility.eq(policy)&
                              combined.model.eq(k)&combined.period_kind.eq('half_year')].set_index('period').loc[periods]
            names=data.population.tolist()
            bounds=ci.loc[ci.model.eq(k)&ci.metric.eq(metric)].set_index('population').loc[names]
            x=np.arange(len(periods))
            ax.plot(x,data[metric].to_numpy(float),marker='o',color=color,label=agency)
            ax.fill_between(x,bounds.ci_low.to_numpy(float),bounds.ci_high.to_numpy(float),alpha=.15,color=color)
        if metric=='risk_gap_pred_minus_observed': ax.axhline(0,color='gray',linestyle='--')
        ax.set(title=label,xticks=np.arange(len(periods)),xticklabels=periods)
        ax.tick_params(axis='x',rotation=55)
        ax.legend()
    fig.suptitle(k+' — matched half-year reliability (95% marginal intervals)')
    fig.tight_layout()
    fig.savefig(OUT/'figures'/f'{k}_temporal_reliability.png',dpi=180,bbox_inches='tight')
    plt.show()
    plt.close(fig)

## Save the aggregate review bundle
Keep local predictions for Notebook 20. Preserve the reviewed inputs and model artifacts. No raw or loan-level records are included in the ZIP.

In [ ]:
assert sha(DATA)==m18['dataset_sha256']
assert sha(p17)==RUN17_HASH
for name in reference_files:
    assert sha(p17.parent/name)==m17['output_sha256'][name]
for k in keys:
    r=m16['models'][k]
    assert sha(COMPANION/r['artifact'])==r['sha256']
    assert sha(COMPANION/r['feature_contract'])==r['feature_contract_sha256']
for name,expected in m16['source_sha256'].items():
    assert sha(ROOT/name)==expected
assert sha(ROOT/'src/fannie_external_evaluation.py')=='0c380257957742615c7aa3d3fb97c70039cae5526151548f8b8cc809da9235c4'
assert sha(COMPANION/'frozen_calibration_bins.json')==m16['output_sha256']['frozen_calibration_bins.json']
manifest=dict(stage='19',status='evaluation_complete_pending_review',dataset_sha256=sha(DATA),
              dataset_release=m18['release_label'],companion_manifest_sha256=sha(COMPANION/'model_manifest.json'),
              notebook17_manifest_sha256=RUN17_HASH,model_keys=keys,predictors=features,
              fannie_primary_loans=len(fannie),fannie_strict_loans=int(fannie.strict.sum()),
              origination_conflicts_retained=482,bootstrap_draws=N_BOOTSTRAP,seed=SEED,
              models_fitted=False,models_updated=False,fico_sensitivity=False,
              risk_gap='predicted minus observed',freddie_reference='Verified Notebook 17 companion results',
              uncertainty='Marginal percentile intervals; independent agency loan draws; nested strict; conditional on fitted models',
              output_sha256={p.relative_to(OUT).as_posix():sha(p) for p in OUT.rglob('*') if p.is_file()})
(OUT/'model_evaluation_manifest.json').write_text(json.dumps(manifest,indent=2))
review=OUT/'notebook_19_final_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix in {'.csv','.json','.png'}:
            z.write(p,p.relative_to(OUT).as_posix())
with zipfile.ZipFile(review) as z:
    assert z.testzip() is None
    assert not any(n.endswith('.parquet') for n in z.namelist())
print('UPLOAD THIS FILE:',review)
print('Keep local predictions:',OUT/'fannie_2018_2023_predictions.parquet')